In [ ]:
import joblib
import random
import optuna
from nn_models   import *
from nn_data     import *
from nn_training import *
from nn_score_and_plot import *
from torch.utils.data            import DataLoader
from sklearn.preprocessing       import StandardScaler
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
DATA_PATH = '/path'
OUT_PATH = '/path'
test_region = 'Central'   

In [ ]:
ts_features       = TS_FEATURES
static_features   = STATIC_FEATURES
features          = ts_features + static_features

df_train = pd.read_csv(DATA_PATH + f'train_{test_region}.csv')
df_val   = pd.read_csv(DATA_PATH + f'val_{test_region}.csv')

In [ ]:
def make_objective(df_tr, df_vl, ts_features, static_features, n_sources, n_fts, n_fstatic):
    """
    Returns an Optuna objective that searches over architecture and training
    hyperparameters using a single train/validation split (no CV).
    df_tr and df_vl must already be split by station to avoid spatial leakage.
    Scalers are fit on df_tr only, fresh inside every trial.
    """
    def objective(trial):

        # Search space
        hts1, hts2, hts3 = suggest_chain(trial, ["hts1", "hts2", "hts3"], low=4, high=7)
        hstt1, hstt2     = suggest_chain(trial, ["hstt1", "hstt2"], low=4, high=6)
        hh1, hh2         = suggest_chain(trial, ["hh1", "hh2"], low=3, high=5)
        hout             = trial.suggest_categorical("hout",  [8, 16, 32])
        hsk              = trial.suggest_categorical("hsk",   [8, 16, 32])
        drop_rate_ts     = trial.suggest_float("drop_rate_ts",  0.1, 0.5, step=0.1)
        drop_rate_stt    = trial.suggest_float("drop_rate_stt", 0.1, 0.5, step=0.1)
        hdrop_rate       = trial.suggest_float("hdrop_rate",    0.1, 0.5, step=0.1)
        lr               = trial.suggest_float("lr",         1e-4, 1e-2, log=True)
        lambda_sim       = trial.suggest_float("lambda_sim", 1e-2, 5e-2, step=0.01)

        df_tr_work = df_tr.copy()
        df_vl_work = df_vl.copy()

        # Fit scalers on train only — no leakage
        fts_sc     = StandardScaler()
        fstatic_sc = StandardScaler()
        t_sc       = StandardScaler()
        df_tr_work[ts_features]     = fts_sc.fit_transform(df_tr_work[ts_features])
        df_tr_work[static_features] = fstatic_sc.fit_transform(df_tr_work[static_features])
        df_tr_work['pm25_filled']   = t_sc.fit_transform(df_tr_work['pm25_filled'].values.reshape(-1, 1))
        df_vl_work[ts_features]     = fts_sc.transform(df_vl_work[ts_features])
        df_vl_work[static_features] = fstatic_sc.transform(df_vl_work[static_features])
        df_vl_work['pm25']          = transform_target_preserve_nan(df_vl_work['pm25'], t_sc)

        tr_groups = prepare_3d_input(df=df_tr_work, ts_cols=ts_features,
                                     static_cols=static_features, target_col='pm25_filled', verbose=False)
        vl_groups = prepare_3d_input(df=df_vl_work, ts_cols=ts_features,
                                     static_cols=static_features, verbose=False)

        tr_tensors = groups_to_tensors(tr_groups, mode='db', reverse_sort=True)
        vl_tensors = groups_to_tensors(vl_groups, mode='db')

        physical_zero_std = -fts_sc.mean_[:n_sources] / fts_sc.scale_[:n_sources]

        # Train
        m = TwoStageDB(
                        n_sources     = n_sources,
                        n_fts         = n_fts,
                        n_fstatic     = n_fstatic,
                        hts1          = hts1,
                        hts2          = hts2,
                        hts3          = hts3,
                        hstt1         = hstt1,
                        hstt2         = hstt2,
                        hout          = hout,
                        hh1           = hh1,
                        hh2           = hh2,
                        hsk           = hsk,
                        drop_rate_ts  = drop_rate_ts,
                        drop_rate_stt = drop_rate_stt,
                        hdrop_rate    = hdrop_rate,
                        threshold     = physical_zero_std,
                        **source_sum_scaler_params(fts_sc, t_sc, n_sources)) 

        opt  = torch.optim.Adam(m.parameters(), lr=lr)
        train_losses, best_val, best_epoch, val_losses = train_TwostageDB(
            m, tr_tensors, opt, MAX_EPOCHS, lambda_sim=lambda_sim,
            vl_tensors=vl_tensors, patience=PATIENCE)

        train_pred_df = collect_TwoStageDB_preds(tr_groups, df_tr, m, 
                                                         t_sc, fts_sc, 
                                                         CMAQ_FEATURES, n_fts, target_col='pm25_filled')
        val_pred_df   = collect_TwoStageDB_preds(vl_groups, df_vl, m, 
                                                         t_sc, fts_sc, 
                                                         CMAQ_FEATURES, n_fts)
        train_score = calculate_score(obs=train_pred_df['pm25_filled'].values, pred=train_pred_df['pred'].values)
        val_pred_eval = val_pred_df.dropna(subset=['pm25']) 
        val_score   = calculate_score(obs=val_pred_eval['pm25'].values, pred=val_pred_eval['pred'].values)
        
        trial.set_user_attr('train_score', train_score.tolist())
        trial.set_user_attr('val_score',   val_score.tolist())
        trial.set_user_attr("hts1", hts1)
        trial.set_user_attr("hts2", hts2)
        trial.set_user_attr("hts3", hts3)
        trial.set_user_attr("hstt1", hstt1)
        trial.set_user_attr("hstt2", hstt2)
        trial.set_user_attr("hh1", hh1)
        trial.set_user_attr("hh2", hh2)
        trial.set_user_attr('best_epoch',  best_epoch)
        trial.set_user_attr('train_losses', train_losses)
        trial.set_user_attr('val_losses',   val_losses)

        print(f"  trial {trial.number}  best_val {best_val:.4f}  best_epoch {best_epoch}",
              flush=True)

        return float(best_val)

    return objective

def optuna_callback(study, trial):
    if trial.state != optuna.trial.TrialState.COMPLETE:
        return
    done = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    tag  = "  ← new best" if trial.value == study.best_value else ""
    best_epoch  = trial.user_attrs.get('best_epoch', '?')
    print(f"[{done:3d}/{N_TRIALS}]  trial {trial.number:3d} "f"Val MSE {trial.value:.4f}  |  best {study.best_value:.4f} "f"best_epoch {best_epoch}{tag}")

In [ ]:
seed_everything()

N_TRIALS    = 25
TIMEOUT     = None   
MAX_EPOCHS  = 200    
PATIENCE    = 20

study = optuna.create_study(
    direction  = "minimize",
    sampler    = optuna.samplers.RandomSampler(seed=88),
    study_name = f"twostage_db_{test_region}")

study.optimize(
    make_objective(df_train, df_val, ts_features, static_features, N_SOURCES, N_FTS, N_FSTATIC),
    n_trials  = N_TRIALS,
    timeout   = TIMEOUT,
    callbacks = [optuna_callback])

In [ ]:
best_trial = study.best_trial

train_curves = best_trial.user_attrs['train_losses']   
val_curves   = best_trial.user_attrs['val_losses']    

fig, ax = plt.subplots(figsize=(6, 4))
fz=16
ax.plot(train_curves, label='Train')
ax.plot(val_curves,   label='Validation')
ax.axvline(best_trial.user_attrs['best_epoch']- 1,
           color='red', linestyle='--', label='Best epoch')
ax.set_title(f'Two-stage DB', fontsize=fz)
ax.set_xlabel('Epoch', fontsize=fz)
ax.tick_params(axis='both', labelsize=fz)
ax.grid(True, linestyle='dotted')
ax.legend(fontsize=fz, loc='upper right')
ax.set_ylabel('Loss', fontsize=fz)
plt.tight_layout()

In [ ]:
print(f"Best CV loss : {study.best_value:.6f}")
print(f"Best params  : {study.best_params}")
print(f"Best epoch   : {study.best_trial.user_attrs['best_epoch']}")
print(f"Best hts1    : {study.best_trial.user_attrs['hts1']}")
print(f"Best hts2    : {study.best_trial.user_attrs['hts2']}")
print(f"Best hts3    : {study.best_trial.user_attrs['hts3']}")
print(f"Best hstt1   : {study.best_trial.user_attrs['hstt1']}")
print(f"Best hstt2   : {study.best_trial.user_attrs['hstt2']}")
print(f"Best hh1     : {study.best_trial.user_attrs['hh1']}")
print(f"Best hh2     : {study.best_trial.user_attrs['hh2']}")
print(f"Train score  : {study.best_trial.user_attrs['train_score']}")
print(f"Val score    : {study.best_trial.user_attrs['val_score']}")

In [ ]:
# joblib.dump(study, OUT_PATH)